# Sequential Card Market — game report

This notebook **is** the report. Point it at a finished run, choose **Run All**, and every table and chart appears below as a cell output. It writes no files and starts or changes no game.

**How to use**

This notebook is a **report template**: the generic copy lives in `report-templates/<definition-id>/<template-id>/`. A report for one run is a copy of it inside that run's folder, made by the report workflow, which sets `RUN_DIR` for you. To run a copy by hand:

1. In the configuration cell below, set `RUN_DIR` to a finished instance directory, for example `~/.prime-env-harness/environments/instances/<instance-id>`.
2. Run all cells (Jupyter: *Run → Run All Cells*; or `jupyter nbconvert --to notebook --execute --inplace game-report.ipynb`).
3. Save the notebook. The outputs are stored in the `.ipynb`, so the charts and tables are still there when you reopen it.

The notebook accepts only finished runs of the current fixed schema (`private_cards_per_player`, `community_card_count`, `trades_per_round`, `settlement_rounds`; two player seats; protocol 2). It stops with an error on any other run. That includes unfinished games, so a live game's private cards are never shown.

**Contents**

| § | Section |
|---|---|
| 0 | Configuration, loading, replay and verification (reused from the existing report generator) |
| 1 | Run settings and seat identities |
| 2 | Final result: P&L, hand wins, cumulative P&L |
| 3 | Per-hand settlements |
| 4 | Where the money is made: by round, by role, by hand |
| 5 | Does decision quality improve as community cards are revealed? |
| 6 | Comparison against fair value (own-information E[V]) |
| 7 | EV versus realized P&L: submitted buy/sell actions, per seat |
| 8 | Both-party exposure, including passive quote fills |
| 9 | Time to decide |
| 10 | Cost and token usage |
| 11 | Action mix and rejections |
| 12 | Complete action table |
| 13 | Public chat |
| 14 | Verification results |
| 15 | Methods and limitations |

## 0. Configuration

This is the only cell you need to edit.

In [ ]:
RUN_DIR = ""  # report-parameters

### 0.1 Setup

Imports, chart styling, and display helpers. Values are kept as exact `Fraction`s throughout and rounded only when shown.

In [ ]:
import html
import json
import math
import statistics
import warnings
from bisect import bisect_left
from collections import Counter, defaultdict
from copy import deepcopy
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

warnings.filterwarnings("ignore", message=".*numexpr.*")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

RUN_DIR = Path(RUN_DIR).expanduser().resolve()

# Seats take categorical slots in roster order; gray is reserved for baselines and references.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
NEUTRAL, GRID, INK2 = "#9a9893", "#e6e5e1", "#52514e"
DIVERGING = LinearSegmentedColormap.from_list("pnl", ["#e34948", "#f0efec", "#2a78d6"])
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": "#c9c8c3", "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False, "lines.linewidth": 2,
})


def num(x, places=2):
    """Display-only rounding with a Unicode minus. Upstream values stay exact."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "–"
    if isinstance(x, Fraction):
        x = int(x) if x.denominator == 1 else float(x)
    text = str(x) if isinstance(x, int) else f"{x:.{places}f}"
    return text.replace("-", "−")


def snum(x, places=2):
    """Signed display: +12, −3.50, 0."""
    text = num(x, places)
    if text.startswith("−") or text == "–" or not text.strip("0."):
        return text
    return "+" + text


def mean(values):
    values = list(values)
    return sum(values, Fraction(0)) / len(values) if values else None


def show(df, places=2, signed=(), escape=True, collapsed=None):
    """Render a DataFrame as a static HTML table (persists in the saved notebook).

    With `collapsed="summary text"` the table is wrapped in an expandable <details> block.
    """
    out = df.copy()
    for col in out.columns:
        if col in signed:
            out[col] = out[col].map(lambda v: snum(v, places) if isinstance(v, (int, float, Fraction)) else v)
        else:
            out[col] = out[col].map(lambda v: num(v, places)
                                    if isinstance(v, (int, float, Fraction)) and not isinstance(v, bool) else v)
    table = out.to_html(index=False, escape=escape, classes="dataframe", border=0)
    if collapsed:
        table = f"<details><summary>{collapsed}</summary>{table}</details>"
    display(HTML(table))


def finish(fig):
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def utc(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def ts_ms(iso):
    return round(datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp() * 1000)


print("RUN_DIR", RUN_DIR)

### 0.1b Template guard

When this notebook sits next to a `template.json` (as it does in an instance's `reports/` folder), the run must be an instance of that template's definition, on a version the template lists. A trial copy (`"trial": true`) is allowed to run on an unlisted version so that the template can be verified against it.

In [ ]:
_tpl_path = Path("template.json")
if not (RUN_DIR / "instance.json").exists():
    raise SystemExit(f"set RUN_DIR to a finished instance directory (no instance.json under {RUN_DIR})")
if _tpl_path.exists():
    _tpl, _inst = json.loads(_tpl_path.read_text()), json.loads((RUN_DIR / "instance.json").read_text())
    if _inst.get("definition_id") != _tpl["definition_id"]:
        raise SystemExit(f"this template is for {_tpl['definition_id']!r}, not {_inst.get('definition_id')!r}")
    if _inst.get("version") not in _tpl["compatible_versions"] and not _tpl.get("trial"):
        raise SystemExit(f"template {_tpl['id']!r} is not verified for version {str(_inst.get('version'))[:12]}")
    print(f"template {_tpl['id']} (revision {_tpl['revision']}) on {_inst.get('definition_id')} "
          f"version {str(_inst.get('version'))[:12]}" + (" [trial]" if _tpl.get("trial") else ""))
else:
    print("no template.json beside this notebook: running standalone, version not checked")

### 0.2 Load and validate the run (reused)

This cell is copied unchanged from the existing generator (`report.ipynb`, step 2). It rejects any run that is unfinished or does not match the fixed schema. All the private-card displays come after this check.

In [ ]:
# Reused verbatim from reporting/sequential-card-market/report.ipynb.
# ---- discover run inputs (read-only) ----------------------------------------------
def _require(path, what):
    if not path.exists():
        raise SystemExit(f"missing {what}: {path}")
    return path

CKPT_PATH = _require(RUN_DIR / "checkpoint.json", "checkpoint.json")
INSTANCE_PATH = _require(RUN_DIR / "instance.json", "instance.json")
RUN_PATH = RUN_DIR / "run.json"

checkpoint = json.loads(CKPT_PATH.read_text())
instance = json.loads(INSTANCE_PATH.read_text())
run_status = json.loads(RUN_PATH.read_text()) if RUN_PATH.exists() else {}

snapshot = checkpoint["snapshot"]
receipts = checkpoint["receipts"]

# ---- reject anything that is not a finished fixed-schema run -----------------------
def _fail(msg):
    raise SystemExit(f"cannot report this run: {msg}")

if checkpoint.get("protocol_version") != 2:
    _fail(f"unsupported protocol_version {checkpoint.get('protocol_version')!r}")
if not checkpoint.get("finished"):
    _fail("run is not finished (checkpoint['finished'] is empty)")

cfg = snapshot.get("config") or {}
required_cfg = {"private_cards_per_player", "community_card_count", "trades_per_round", "settlement_rounds"}
missing = required_cfg - set(cfg)
if missing:
    _fail(f"fixed-schema config missing fields: {sorted(missing)}")
if any(type(v) is not int or v < 1 for v in cfg.values()):
    _fail(f"config values must be positive integers: {cfg}")

players = list(snapshot.get("players") or [])
if len(players) != 2 or len(set(players)) != 2:
    _fail(f"expected exactly two distinct player seats, got {players}")
if snapshot.get("phase") != "finished":
    _fail(f"snapshot phase is {snapshot.get('phase')!r}, expected 'finished'")
history = snapshot.get("history") or []
if len(history) != cfg["settlement_rounds"]:
    _fail(f"settled {len(history)} hands, config requires {cfg['settlement_rounds']}")

seat_meta = {s["id"]: s for s in (instance.get("roster", {}) or {}).get("seats", []) if s.get("role") == "player"}
seats = [seat_meta.get(p, {"id": p, "role": "player"}) for p in players]

print(f"finished fixed-schema run: {len(players)} seats, {len(history)} hands, "
      f"{len(receipts)} receipts")
print("config:", cfg)
print("seats :", [s.get("id") for s in seats])

### 0.3 Replay the recorded deals and receipts (reused)

This cell is copied from the existing generator (step 3). The only change: if the replay fails, the run is rejected as unsupported with a clear message instead of a bare `AssertionError`. It starts from the first deal, replays every receipt in sequence order, and checks each accepted action against the rules. It also confirms that rejected attempts left the state unchanged and that each hand's value and P&L match the recorded settlement. The rebuilt final engine state must equal `checkpoint.snapshot` exactly. The exact `Fraction` EV accounting (`fair_value`, `exposure`, `totals`) is defined here as well.

In [ ]:
# Reused from reporting/sequential-card-market/report.ipynb. Only change: a replay or snapshot
# mismatch is reported as an unsupported run instead of a bare AssertionError.
# ---- deterministic replay of the recorded deals and receipts ----------------------
# Standard 52-card ranks (A=1 .. K=13). Derived values only; nothing run-specific.
RANKS = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
HAND_CARDS = 2 * cfg["private_cards_per_player"] + cfg["community_card_count"]
FULL_RANK_SUM = sum(RANKS.index(r) + 1 for r in RANKS) * 4   # 364 for a full deck
FIRST, SECOND = players[0], players[1]


def rank(card):
    return RANKS.index(card[:-1]) + 1


def other(p):
    return next(q for q in players if q != p)


def opener(hand_index, round_index):
    """The environment's opening rule: priority alternates each round and flips each hand."""
    return players[(hand_index + round_index) % 2]


def fair_value(cards_known):
    """Exact expected settlement value from the cards a seat knows at a fill.

    Known cards are the seat's own private cards plus community cards revealed before
    the fill. k of the hand's cards are known with rank sum S; the rest are drawn
    without replacement, so E[V] = S + (HAND_CARDS - k) * (FULL_RANK_SUM - S) / (52 - k).
    Returns the exact expectation plus the min/max possible settlement values.
    """
    z = sum(rank(c) for c in cards_known)
    n = len(cards_known)
    deck = sorted(rank(r + s) for r in RANKS for s in "SHDC" if r + s not in cards_known)
    remain = HAND_CARDS - n
    return (Fraction(z) + Fraction(remain * (FULL_RANK_SUM - z), 52 - n),
            z + sum(deck[:remain]), z + sum(deck[-remain:]))


def describe(event):
    who, kind, label = event["actor"], event["kind"], f"R{event['round']}"
    if kind == "buy":
        return f"{label} {who} buys at {event['price']} from {event['counterparty']}"
    if kind == "sell":
        return f"{label} {who} sells at {event['price']} to {event['counterparty']}"
    verb = "quotes" if kind == "open_market" else "improves to"
    return f"{label} {who} {verb} {event['bid']}/{event['ask']}"


def exposure(seq, p, actor, side, price, h, rd):
    f, lo, hi = fair_value(history[h - 1]["cards"][p] + history[h - 1]["board"][:rd - 1])
    v = history[h - 1]["value"]
    ev = f - price if side == "buy" else price - f
    real = v - price if side == "buy" else price - v
    return dict(sequence=seq, hand=h, round=rd, participant=p, initiator=actor,
                role="initiator" if p == actor else "quote_provider", side=side, price=price,
                expected_value=float(f), expected_value_exact=str(f),
                possible_value_min=lo, possible_value_max=hi, settlement_value=v,
                ev=float(ev), ev_exact=str(ev), real_pnl=real,
                ev_minus_pnl=float(ev - real), ev_minus_pnl_exact=str(ev - real))


def replay():
    """Rebuild committed engine state from the first deal and the sorted receipts."""
    state = {"players": list(players), "revision": 0, "phase": "trading", "config": cfg,
             "settlement_round": 0, "total_pnl": {p: 0 for p in players}, "history": [],
             "events": [], "event_seq": 0}

    def deal(hand_index):
        h = history[hand_index]
        state.update({"hands": deepcopy(h["cards"]), "community": h["board"][:], "round": 0,
                      "revealed": [], "market": None, "trades_this_round": 0, "trades": [],
                      "positions": {p: 0 for p in players}, "cash": {p: 0 for p in players},
                      "settlement": None, "phase": "trading",
                      "active": opener(hand_index, 0), "events": state["events"][-30:]})

    def record(actor, kind, **detail):
        state["event_seq"] += 1
        settled = state["phase"] in ("settled", "finished")
        state["events"].append({"seq": state["event_seq"],
                                "hand": state["settlement_round"] if settled else state["settlement_round"] + 1,
                                "round": cfg["community_card_count"] if settled else state["round"] + 1,
                                "actor": actor, "kind": kind, **detail})

    def opening_priority():
        if state["phase"] == "finished":
            return None
        hi = state["settlement_round"]
        ri = 0 if state["phase"] == "settled" else state["round"]
        return {"hand": hi + 1, "hand_opener": opener(hi, 0), "round_opener": opener(hi, ri)}

    def legal(p):
        if state["phase"] == "settled":
            return ["next_hand"] if p == players[0] else []
        if state["phase"] != "trading" or state["active"] != p:
            return []
        m = state["market"]
        if m is None:
            return ["open_market"]
        out = []
        if m["ask_owner"] != p:
            out.append("buy")
        if m["bid_owner"] != p:
            out.append("sell")
        if m["ask"] - m["bid"] > 1:
            out.append("improve_market")
        return out

    def view(p):
        v = {"you": p, "opponent": other(p), "player_order": list(players), "phase": state["phase"],
             "opening_priority": opening_priority(),
             "progress": {"hands_settled": state["settlement_round"], "hands_total": cfg["settlement_rounds"],
                          "trading_round": min(state["round"] + 1, cfg["community_card_count"]),
                          "rounds_per_hand": cfg["community_card_count"],
                          "trades_this_round": state["trades_this_round"],
                          "trades_per_round": cfg["trades_per_round"]},
             "community_cards": list(state["revealed"]), "your_private_cards": list(state["hands"][p]),
             "active_player": state["active"], "market": deepcopy(state["market"]),
             "positions": dict(state["positions"]), "cash": dict(state["cash"]),
             "total_pnl": dict(state["total_pnl"]), "available_actions": legal(p),
             "recent_events": deepcopy(state["events"][-30:])}
        if state["event_seq"] > len(v["recent_events"]):
            v["earlier_events_omitted"] = state["event_seq"] - len(v["recent_events"])
        hist = state["history"]
        v["past_hands"] = deepcopy(hist[-10:])
        if len(hist) > 10:
            older = hist[:-10]
            v["earlier_hands_summary"] = {"hands": len(older),
                                          "pnl": {q: sum(hh["pnl"][q] for hh in older) for q in players}}
        if state["phase"] in ("settled", "finished"):
            v["all_private_cards"] = deepcopy(state["hands"])
            v["settlement"] = deepcopy(state["settlement"])
        if state["phase"] == "finished":
            v["winners"] = [q for q in players if state["total_pnl"][q] == max(state["total_pnl"].values())]
        return v

    actions, chats, actorrows, allrows = [], [], [], []
    deal(0)
    for rid, r in sorted(receipts.items(), key=lambda x: x[1]["sequence"]):
        req, method, seq = r["request"], r["request"]["method"], r["sequence"]
        if method == "initialize":
            continue
        p = req.get("participant")
        if method == "handle_input":
            text = req["params"]["text"]
            record(p, "chat", text=text[:300], **({"truncated": True} if len(text) > 300 else {}))
            chats.append({"sequence": seq, "participant": p, "text": text, "success": r["success"],
                          "state_before": view(p), "state_after": view(p)})
            continue
        assert method == "invoke", method
        a, args = req["params"]["action"], req["params"]["arguments"]
        before, eng_before = view(p), deepcopy(state)
        f, lo, hi = fair_value(state["hands"][p] + state["revealed"])
        row = dict(sequence=seq, receipt_id=rid, hand_before=before["progress"]["hands_settled"] + 1,
                   round_before=state["round"] + 1, actor=p, action=a, arguments=args,
                   success=r["success"], state_before=before, committed_engine_before=eng_before,
                   own_card_expected_value=float(f), own_card_expected_value_exact=str(f),
                   possible_value_min=lo, possible_value_max=hi,
                   error=r.get("error"), result=r.get("result"))
        if r["success"]:
            assert a in legal(p), (seq, a, legal(p))
            if a == "next_hand":
                assert not args
                deal(state["settlement_round"])
                event = f"{p} started hand {state['settlement_round'] + 1}."
            elif a in ("open_market", "improve_market"):
                bid, ask = args["bid"], args["ask"]
                assert set(args) == {"bid", "ask"} and type(bid) is int and type(ask) is int and 0 <= bid < ask
                if a == "improve_market":
                    prev = state["market"]
                    assert not (bid < prev["bid"] or ask > prev["ask"] or (bid == prev["bid"] and ask == prev["ask"]))
                state["market"] = {"bid": bid, "ask": ask, "bid_owner": p, "ask_owner": p}
                state["active"] = other(p)
                record(p, a, bid=bid, ask=ask)
                event = f"{p} {a}: bid {bid}, ask {ask}."
            else:
                assert not args
                m = state["market"]
                provider = m["ask_owner"] if a == "buy" else m["bid_owner"]
                price = m["ask"] if a == "buy" else m["bid"]
                buyer, seller = (p, provider) if a == "buy" else (provider, p)
                er = exposure(seq, p, p, a, price, row["hand_before"], row["round_before"])
                actorrows.append(er)
                allrows.append(dict(er))
                allrows.append(exposure(seq, provider, p, "sell" if a == "buy" else "buy",
                                        price, row["hand_before"], row["round_before"]))
                row["execution_edge"] = er
                state["positions"][buyer] += 1
                state["positions"][seller] -= 1
                state["cash"][buyer] -= price
                state["cash"][seller] += price
                state["trades_this_round"] += 1
                state["trades"].append({"round": state["round"], "buyer": buyer, "seller": seller, "price": price})
                state["market"] = None
                record(p, a, price=price, counterparty=provider)
                event = f"{buyer} bought one contract from {seller} at {price}."
                if state["trades_this_round"] == cfg["trades_per_round"]:
                    state["revealed"].append(state["community"][state["round"]])
                    state["events"][-1]["revealed"] = state["revealed"][-1]
                    state["round"] += 1
                    event += f" Round {state['round']} complete; revealed {state['revealed'][-1]}."
                    state["trades_this_round"] = 0
                    if state["round"] == cfg["community_card_count"]:
                        value = sum(rank(c) for hand in state["hands"].values() for c in hand)
                        value += sum(rank(c) for c in state["community"])
                        pnl = {q: state["cash"][q] + state["positions"][q] * value for q in players}
                        assert value == history[state["settlement_round"]]["value"]
                        assert pnl == history[state["settlement_round"]]["pnl"]
                        assert len(state["trades"]) == cfg["community_card_count"] * cfg["trades_per_round"]
                        for q in players:
                            state["total_pnl"][q] += pnl[q]
                        state["settlement_round"] += 1
                        state["settlement"] = {"value": value, "pnl": pnl,
                                               "winners": [q for q in players if pnl[q] == max(pnl.values())]}
                        acts = [describe(e) for e in state["events"]
                                if e["hand"] == state["settlement_round"] and e["kind"] != "chat"]
                        state["history"].append({"hand": state["settlement_round"], "value": value, "pnl": pnl,
                                                 "cards": deepcopy(state["hands"]), "board": list(state["community"]),
                                                 "actions": acts})
                        state["phase"] = "finished" if state["settlement_round"] == cfg["settlement_rounds"] else "settled"
                        state["active"] = None if state["phase"] == "finished" else players[0]
                    else:
                        state["active"] = opener(state["settlement_round"], state["round"])
            state["revision"] += 1
            assert r["result"] == {"event": event}, (seq, r["result"], event)
            assert r["progress"]["revision"] == state["revision"]
        else:
            assert state == eng_before
        row.update(state_after=view(p), committed_engine_after=deepcopy(state))
        actions.append(row)
    return state, actions, chats, actorrows, allrows


def totals(rows):
    sums = {p: {"ev": Fraction(0), "real_pnl": 0, "count": 0} for p in players}
    residual = dict.fromkeys(players, Fraction(0))
    for r in rows:
        p = r["participant"]
        sums[p]["ev"] += Fraction(r["ev_exact"])
        sums[p]["real_pnl"] += r["real_pnl"]
        sums[p]["count"] += 1
        residual[p] += Fraction(r["ev_minus_pnl_exact"])
        r["cumulative_ev_minus_pnl"] = float(residual[p])
        r["cumulative_ev_minus_pnl_exact"] = str(residual[p])
    return {p: dict(ev=float(t["ev"]), ev_exact=str(t["ev"]), real_pnl=t["real_pnl"], count=t["count"],
                    ev_minus_pnl=float(t["ev"] - t["real_pnl"]),
                    ev_minus_pnl_exact=str(t["ev"] - t["real_pnl"])) for p, t in sums.items()}


# ---- reconcile the reconstruction against the recorded checkpoint ------------------
try:
    state, actions, chats, actorrows, allrows = replay()
    assert state == snapshot, [(k, state[k], snapshot[k]) for k in state if state[k] != snapshot[k]]
except AssertionError as exc:
    raise SystemExit("cannot report this run: its receipts do not replay under the current fixed rules "
                     f"(an older or different environment version?); first failed check: {str(exc)[:200]}") from None
assert len(actions) == sum(1 for r in receipts.values() if r["request"]["method"] == "invoke")
assert len(chats) == sum(1 for r in receipts.values() if r["request"]["method"] == "handle_input")
assert len(actorrows) == sum(a["success"] and a["action"] in ("buy", "sell") for a in actions)
assert len(allrows) == 2 * len(actorrows)

at, bt = totals(actorrows), totals(allrows)

print(f"replay OK: {len(actions)} invoke attempts, {len(chats)} public say calls, "
      f"{len(actorrows)} fills, final P&L {dict(state['total_pnl'])}")
print("actor totals   :", {p: round(at[p]["ev"], 6) for p in players})
print("all-party totals:", {p: round(bt[p]["ev"], 6) for p in players})

### 0.4 Verify seat wake states (reused)

This cell is copied unchanged from the existing generator (step 4). Every wake state in each seat's transcript is checked against the replayed deals. The check uses only state fields; no reasoning or assistant text is read into the report.

In [ ]:
# Reused verbatim from reporting/sequential-card-market/report.ipynb.
# ---- read seat transcripts read-only (for wake verification only) ------------------
# Only the two seat conversation logs and the wake custom messages are read. Raw
# reasoning, assistant text, and tool arguments are never copied into the bundle.
seat_dirs = {}
for p in players:
    cand = RUN_DIR / "seats" / p / "conversation.jsonl"
    seat_dirs[p] = cand if cand.exists() else None
print("seat transcripts:", {p: (str(d) if d else None) for p, d in seat_dirs.items()})

transcript_records = {}
for p, path in seat_dirs.items():
    recs = []
    if path is not None:
        for line in path.read_text().splitlines():
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    transcript_records[p] = recs

# ---- verify every wake state against the replayed deals ----------------------------
wakecount = 0
wake_problems = []
for p, recs in transcript_records.items():
    for record in recs:
        if record.get("customType") != "environment_wake":
            continue
        try:
            data = json.loads(record["content"].split("\n", 1)[1])
        except (KeyError, IndexError, ValueError):
            continue
        v = data.get("state")
        if not isinstance(v, dict):
            continue
        hs = v["progress"]["hands_settled"]
        h = hs + 1 if v["phase"] == "trading" else hs
        checks = [
            v.get("you") == p,
            v.get("your_private_cards") == history[h - 1]["cards"][p],
            v.get("community_cards") == history[h - 1]["board"][:len(v.get("community_cards", []))],
            v.get("total_pnl") == {q: sum(z["pnl"][q] for z in history[:hs]) for q in players},
        ]
        if v["phase"] in ("settled", "finished"):
            checks.append(v.get("all_private_cards") == history[h - 1]["cards"])
            checks.append(v.get("settlement") == {"value": history[h - 1]["value"], "pnl": history[h - 1]["pnl"],
                                                  "winners": [q for q in players if history[h - 1]["pnl"][q] ==
                                                              max(history[h - 1]["pnl"].values())]})
        if all(checks):
            wakecount += 1
        else:
            wake_problems.append((p, record.get("id"), v["phase"], hs))
if wake_problems:
    raise SystemExit(f"wake verification failed for {len(wake_problems)} records: {wake_problems[:3]}")
print(f"verified {wakecount} transcript wake states against the replayed deals")

### 0.5 Read timing, token, and cost metadata from the seat transcripts

For each seat transcript, this cell reads only:

* timestamps of wakes and of model requests and responses,
* token-usage and cost fields,
* the tool name, arguments, and outcome of each tool call.

Assistant text and reasoning are never read into a variable that gets displayed.

Each tool call is paired with its environment receipt in order. Calls that never reached the environment have no receipt and are counted separately. These are stale-view calls (`stale_view`: the state had changed since the model last looked), unknown tool names, and host-side errors. The cell stops if any receipt has no matching call.

In [ ]:
seat_receipts = {p: [r for _, r in sorted(receipts.items(), key=lambda x: x[1]["sequence"])
                     if r["request"]["method"] in ("invoke", "handle_input")
                     and r["request"].get("participant") == p] for p in players}
action_by_seq = {a["sequence"]: a for a in actions}
chat_by_seq = {c["sequence"]: c for c in chats}

# Commit time of each environment operation, from the public table log.
TABLE_PATH = RUN_DIR / "table.jsonl"
commit_ms = {}
if TABLE_PATH.exists():
    for line in TABLE_PATH.read_text().splitlines():
        if line.strip():
            ev = json.loads(line)
            if ev.get("op_seq") is not None:
                commit_ms[ev["op_seq"]] = min(ev["timestamp"], commit_ms.get(ev["op_seq"], ev["timestamp"]))


def receipt_context(seq):
    """One-based hand and trading round in play when a receipt was submitted."""
    if seq in action_by_seq:
        return action_by_seq[seq]["hand_before"], action_by_seq[seq]["round_before"]
    prog = chat_by_seq[seq]["state_before"]["progress"]
    return prog["hands_settled"] + 1, prog["trading_round"]


llm_calls, tool_calls, wakes = [], [], []
telemetry_seats = []
seat_meta_rt = {}
for p in players:
    recs = transcript_records[p]
    if not recs:
        continue
    telemetry_seats.append(p)
    results = {r["message"]["toolCallId"]: r["message"] for r in recs
               if (r.get("message") or {}).get("role") == "toolResult"}
    queue, qi, wake = seat_receipts[p], 0, None
    for rec in recs:
        if rec.get("type") == "model_change":
            seat_meta_rt.setdefault(p, {})["provider/model"] = f"{rec.get('provider')}/{rec.get('modelId')}"
        if rec.get("type") == "service_tier_change":
            seat_meta_rt.setdefault(p, {})["service tier"] = rec.get("serviceTier")
        if rec.get("customType") == "environment_wake":
            wake = {"seat": p, "t": ts_ms(rec["timestamp"]), "first_action_ms": None}
            wakes.append(wake)
            continue
        msg = rec.get("message") or {}
        if msg.get("role") != "assistant":
            continue
        usage = msg.get("usage") or {}
        start, end = msg["timestamp"], ts_ms(rec["timestamp"])
        calls = [c for c in msg.get("content", []) if c.get("type") == "toolCall"]
        llm_calls.append(dict(seat=p, start_ms=start, end_ms=end, latency_s=(end - start) / 1000,
                              input=usage.get("input", 0), output=usage.get("output", 0),
                              cache_read=usage.get("cacheRead", 0), cache_write=usage.get("cacheWrite", 0),
                              total_tokens=usage.get("totalTokens", 0),
                              cost=(usage.get("cost") or {}).get("total", 0),
                              stop=msg.get("stopReason"), n_tools=len(calls)))
        for c in calls:
            name = c["name"]
            res = results.get(c["id"])
            text = res["content"][0]["text"] if res and res.get("content") else ""
            try:
                parsed = json.loads(text)
            except ValueError:
                parsed = None
            harness_rejection = (parsed or {}).get("result", {}).get("rejected") \
                if isinstance(parsed, dict) and isinstance(parsed.get("result"), dict) else None
            row = dict(seat=p, tool=name, action="say" if name == "say" else name.removeprefix("env_"),
                       latency_s=(end - start) / 1000, end_ms=end, batched=len(calls) > 1, sequence=None)
            env_call = name == "say" or name.startswith("env_")
            host_error = res is not None and res.get("isError") and "environment rejected operation" not in text
            if not env_call or host_error:
                # Unknown tool names and host-side failures never reach the environment, so they have no receipt.
                row.update(outcome="harness: unknown tool" if not env_call else "harness: host error",
                           hand=None, round=None)
            elif harness_rejection:
                prog = parsed.get("state", {}).get("progress", {})
                row.update(outcome=f"harness: {harness_rejection}",
                           hand=prog.get("hands_settled", 0) + 1, round=prog.get("trading_round"))
            else:
                assert qi < len(queue), (p, "tool call with no remaining receipt", name)
                r = queue[qi]
                qi += 1
                req = r["request"]
                if name == "say":
                    assert req["method"] == "handle_input" and req["params"]["text"] == c["arguments"].get("text"), (p, r["sequence"])
                    outcome = "say"
                else:
                    assert req["method"] == "invoke" and req["params"]["action"] == row["action"] \
                        and req["params"]["arguments"] == (c.get("arguments") or {}), (p, r["sequence"], name)
                    outcome = "accepted" if r["success"] else "rejected"
                hand, rnd = receipt_context(r["sequence"])
                row.update(outcome=outcome, sequence=r["sequence"], hand=hand, round=rnd)
                if wake is not None and wake["first_action_ms"] is None and name != "say":
                    wake["first_action_ms"] = end
            tool_calls.append(row)
    assert qi == len(queue), (p, f"{len(queue) - qi} receipts were not matched to a transcript tool call")

calls_df = pd.DataFrame(llm_calls)
tools_df = pd.DataFrame(tool_calls)

# Assign every model request to the hand in progress when it finished (hand end = commit of its last fill).
hand_end_ms = []
for h in range(1, len(history) + 1):
    last_fill = max(r["sequence"] for r in actorrows if r["hand"] == h)
    hand_end_ms.append(commit_ms.get(last_fill))
if llm_calls and all(hand_end_ms):
    calls_df["hand"] = [min(bisect_left(hand_end_ms, t) + 1, len(history)) for t in calls_df["end_ms"]]

print(f"transcripts read for {telemetry_seats}: {len(llm_calls)} model requests, {len(tool_calls)} tool calls, "
      f"{sum(t['sequence'] is not None for t in tool_calls)} matched to receipts, "
      f"{sum(str(t['outcome']).startswith('harness') for t in tool_calls)} stopped by the harness before the environment")

### 0.6 Derived tables

These tables come from the replay above and are shared by the sections that follow. Every total is computed with exact fractions.

In [ ]:
SEAT_COLOR = {p: SERIES[i % len(SERIES)] for i, p in enumerate(players)}
N_HANDS, ROUNDS = len(history), cfg["community_card_count"]
A, B = players
MARK = {"buy": "^", "sell": "v"}


def round_name(r):
    shown = r - 1
    return f"R{r} ({shown} board card{'' if shown == 1 else 's'} shown)"


def F(row, key):
    return Fraction(row[key])


# Hand-level results.
cum = dict.fromkeys(players, 0)
curve = []
wins, ties = dict.fromkeys(players, 0), 0
for h in history:
    for p in players:
        cum[p] += h["pnl"][p]
    top = max(h["pnl"].values())
    leaders = [p for p in players if h["pnl"][p] == top]
    if len(leaders) == 1:
        wins[leaders[0]] += 1
    else:
        ties += 1
    curve.append(dict(hand=h["hand"], value=h["value"], winner=leaders[0] if len(leaders) == 1 else "tie",
                      **{f"{p}_hand": h["pnl"][p] for p in players}, **{f"{p}_cum": cum[p] for p in players}))

# Quotes (accepted open/improve) and which quote each fill traded against.
quotes, live = [], None
for a in actions:
    if not a["success"]:
        continue
    if a["action"] in ("open_market", "improve_market"):
        bid, ask = a["arguments"]["bid"], a["arguments"]["ask"]
        quotes.append(dict(sequence=a["sequence"], seat=a["actor"], hand=a["hand_before"], round=a["round_before"],
                           kind=a["action"], bid=bid, ask=ask, mid=Fraction(bid + ask, 2), spread=ask - bid,
                           E=Fraction(a["own_card_expected_value_exact"]), V=history[a["hand_before"] - 1]["value"],
                           filled=False, taken_by=None))
        live = len(quotes) - 1
    elif a["action"] in ("buy", "sell"):
        quotes[live].update(filled=True, taken_by=a["action"])
        live = None
    else:
        live = None
for q in quotes:
    q["mid_err"] = abs(q["mid"] - q["V"])      # quote midpoint vs eventual settlement value
    q["fv_err"] = abs(q["E"] - q["V"])          # the quoter's own fair value vs settlement value
    q["mid_vs_E"] = q["mid"] - q["E"]           # quote skew relative to own fair value
    q["straddles_E"] = q["bid"] <= q["E"] <= q["ask"]

fills = [dict(r, E=F(r, "expected_value_exact"), EV=F(r, "ev_exact"), RES=F(r, "ev_minus_pnl_exact"),
              CRES=F(r, "cumulative_ev_minus_pnl_exact")) for r in actorrows]
expos = [dict(r, E=F(r, "expected_value_exact"), EV=F(r, "ev_exact")) for r in allrows]

opens_by_seat = {p: sum(a["success"] and a["action"] == "open_market" and a["actor"] == p for a in actions) for p in players}
n_observe = sum(r["request"]["method"] == "observe" for r in receipts.values())

assert all(sum(e["real_pnl"] for e in expos if e["participant"] == p) == state["total_pnl"][p] for p in players)
print(f"{N_HANDS} hands, {len(fills)} fills, {len(quotes)} accepted quotes, {len(expos)} exposures; "
      f"both-party realized totals reconcile to final P&L {dict(state['total_pnl'])}")

## 1. Run settings and seat identities

Every value below comes from the saved run files: `instance.json` and `run.json`, the checkpoint config, the seat transcripts' model and service-tier records, and the table log.

In [ ]:
start_ms = run_status.get("started_at") or instance.get("created_at")
end_ms = max(commit_ms.values()) if commit_ms else None
policy = instance.get("policy", {})
limits = (instance.get("roster") or {}).get("limits", {})
settings = [
    ("Run name", instance.get("name") or "Sequential Card Market"),
    ("Instance", instance.get("id", RUN_DIR.name)),
    ("Definition", f"{instance.get('definition_id')} (version {str(instance.get('version', ''))[:12]}…)"),
    ("Status", f"{run_status.get('phase', snapshot['phase'])}: {run_status.get('reason', '')}"),
    ("Started", utc(start_ms) if start_ms else "not recorded"),
    ("Last environment event", utc(end_ms) if end_ms else "not recorded"),
    ("Wall-clock duration", f"{(end_ms - start_ms) / 60000:.1f} min" if start_ms and end_ms else "not recorded"),
    ("Hands", f"{N_HANDS} (settlement_rounds = {cfg['settlement_rounds']})"),
    ("Private cards per player", cfg["private_cards_per_player"]),
    ("Community cards per hand", f"{ROUNDS} (one trading round before each reveal)"),
    ("Contracts per round", cfg["trades_per_round"]),
    ("Cards in the settlement value", f"{HAND_CARDS} (all private and community cards; A=1 … K=13)"),
    ("Execution policy", ", ".join(f"{k}={v}" for k, v in policy.items())),
    ("Seat limits", ", ".join(f"{k}={v}" for k, v in limits.items())),
    ("Environment operations", f"{len(receipts) - 1} receipts after initialization ({len(actions)} invokes, "
                               f"{len(chats)} public say calls, {n_observe} observe calls)"),
]
show(pd.DataFrame(settings, columns=["Setting", "Value"]))

seat_rows = []
for i, s in enumerate(seats):
    p = s["id"]
    seat_rows.append({
        "Seat": p, "Roster order": i + 1, "Role": s.get("role"), "Occupant": s.get("occupant"),
        "Model (roster)": s.get("model"), "Thinking": s.get("thinking"),
        "Provider/model (transcript)": seat_meta_rt.get(p, {}).get("provider/model", "–"),
        "Service tier": seat_meta_rt.get(p, {}).get("service tier", "–"),
        "Wakes (run.json)": (run_status.get("wakes") or {}).get(p, "–"),
        "Round-1 opener in": f"{sum(opener(h, 0) == p for h in range(N_HANDS))} hands",
        "Controls next_hand": "yes" if p == players[0] else "no",
    })
show(pd.DataFrame(seat_rows))

## 2. Final result

The final P&L is the sum of the settled hand P&L. A hand win goes to the seat with the higher P&L for that hand; equal P&L counts as a tie. Cash and positions reset at the start of each hand.

In [ ]:
summary = pd.DataFrame([{
    "Seat": p, "Final P&L": state["total_pnl"][p], "Hands won": wins[p], "Ties": ties,
    "Best hand": max(h["pnl"][p] for h in history), "Worst hand": min(h["pnl"][p] for h in history),
    "Mean hand P&L": Fraction(state["total_pnl"][p], N_HANDS),
    "Fills as taker": sum(f["participant"] == p for f in fills),
    "Fills as quote owner": sum(e["participant"] == p and e["role"] == "quote_provider" for e in expos),
    "Markets opened": opens_by_seat[p],
} for p in players])
show(summary, signed=("Final P&L", "Best hand", "Worst hand", "Mean hand P&L"))
display(Markdown(
    f"**{N_HANDS}/{cfg['settlement_rounds']} hands settled.** " +
    "; ".join(f"{p} **{snum(state['total_pnl'][p])}**" for p in players) +
    f". Hand wins {' – '.join(str(wins[p]) for p in players)} ({' – '.join(players)}), ties {ties}. "
    f"{len(fills)} fills; markets opened {' / '.join(str(opens_by_seat[p]) for p in players)}."))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
hands = [0] + [c["hand"] for c in curve]
for p in players:
    ys = [0] + [c[f"{p}_cum"] for c in curve]
    ax1.plot(hands, ys, color=SEAT_COLOR[p], label=p)
    ax1.annotate(f"{p} {snum(ys[-1])}", (hands[-1], ys[-1]), xytext=(6, 0), textcoords="offset points",
                 va="center", fontsize=9, color=INK2)
ax1.axhline(0, color=NEUTRAL, linewidth=0.8)
ax1.set_title("Cumulative settled P&L")
ax1.set_ylabel("P&L")
ax1.legend(loc="upper left")
bar_colors = [SEAT_COLOR[c["winner"]] if c["winner"] != "tie" else NEUTRAL for c in curve]
ax2.bar([c["hand"] for c in curve], [c[f"{A}_hand"] for c in curve], color=bar_colors, width=0.7)
ax2.axhline(0, color=NEUTRAL, linewidth=0.8)
ax2.set_title(f"P&L by hand, shown from {A}'s side (bar colour = hand winner)")
ax2.set_ylabel(f"{A} hand P&L")
ax2.set_xlabel("Hand")
ax2.set_xlim(0, N_HANDS + 3)
finish(fig)

### Highlights

These sentences are generated from the analyses in §4–§11 and describe this match only. They are descriptive, not causal.

In [ ]:
winner = max(players, key=lambda p: state["total_pnl"][p])
loser = other(winner)
r_pnl = {r: sum(e["real_pnl"] for e in expos if e["participant"] == winner and e["round"] == r) for r in range(1, ROUNDS + 1)}
best_r = max(r_pnl, key=r_pnl.get)
role_pnl = {(p, role): sum(e["real_pnl"] for e in expos if e["participant"] == p and e["role"] == role)
            for p in players for role in ("initiator", "quote_provider")}
lines = [
    f"**{winner}** finished {snum(state['total_pnl'][winner])} and won {wins[winner]} of {N_HANDS} hands.",
    f"By round, {winner} made the most in **{round_name(best_r)}**: {snum(r_pnl[best_r])} of {snum(state['total_pnl'][winner])} "
    f"(" + ", ".join(f"R{r} {snum(v)}" for r, v in r_pnl.items()) + ").",
    "By role: " + "; ".join(f"{p} {snum(role_pnl[(p, 'initiator')])} as taker, {snum(role_pnl[(p, 'quote_provider')])} as quote owner"
                            for p in players) + ".",
]
for p in players:
    qs = [q for q in quotes if q["seat"] == p]
    tk = [f for f in fills if f["participant"] == p]
    if qs:
        skew = mean(q["mid_vs_E"] for q in qs)
        lines.append(f"{p}'s quote midpoints sat on average {num(abs(skew))} {'below' if skew < 0 else 'above'} its own fair value; "
                     f"{100 * sum(q['straddles_E'] for q in qs) / len(qs):.0f}% of its quotes contained E[V]. "
                     f"{sum(f['EV'] > 0 for f in tk)} of its {len(tk)} taker trades were on the side its fair value favoured.")
if len(calls_df):
    act_ = tools_df[tools_df["action"] != "say"]
    lines.append("Median model response time per action: " +
                 ", ".join(f"{p} {act_[act_['seat'] == p]['latency_s'].median():.2f} s" for p in telemetry_seats) +
                 f". The harness stopped {int(tools_df['outcome'].astype(str).str.startswith('harness').sum())} calls before they reached the environment.")
    lines.append(f"Recorded model cost: **${calls_df['cost'].sum():.4f}** in total (" +
                 ", ".join(f"{p} ${calls_df[calls_df['seat'] == p]['cost'].sum():.4f}" for p in telemetry_seats) +
                 f"), {int(calls_df['total_tokens'].sum()):,} tokens.")
display(Markdown("\n".join(f"- {line}" for line in lines)))

## 3. Per-hand settlements

Each row is one settled hand. The cards were private while the hand was being played; they became public when it settled. `Fills` lists every execution as *round, taker, side @ price*. `Value` is the settlement value, the rank sum of every card in the hand.

In [ ]:
fills_by_hand = defaultdict(list)
for f in fills:
    fills_by_hand[f["hand"]].append(f"R{f['round']} {f['participant']} {f['side']} @{f['price']}")
rows = []
for h, c in zip(history, curve):
    rows.append({"Hand": h["hand"], "R1 opener": opener(h["hand"] - 1, 0),
                 **{f"{p} cards": " ".join(h["cards"][p]) for p in players},
                 "Board": " ".join(h["board"]), "Value": h["value"], "Fills": "; ".join(fills_by_hand[h["hand"]]),
                 **{f"{p} hand": h["pnl"][p] for p in players}, **{f"{p} cumulative": c[f"{p}_cum"] for p in players},
                 "Winner": c["winner"]})
show(pd.DataFrame(rows), signed=tuple(f"{p} {k}" for p in players for k in ("hand", "cumulative")))

## 4. Where the money is made

Each fill creates two exposures: the **taker**, who sent the buy or sell, and the **quote owner**, whose resting bid or ask was hit. Both are valued at the hand's settlement, so each seat's exposures add up exactly to its final P&L.

A hand has one trading round before each community card is revealed:

* **R1** trades with no community cards showing (only the seat's own private cards).
* **R2** trades after the first community card is revealed.
* **R3** trades after the second.

The last card is revealed after R3 and the hand settles right away. *"Where do agents make money: first, second, or third card?"* is therefore answered per trading round below.

In [ ]:
by_round = []
for p in players:
    for r in range(1, ROUNDS + 1):
        rows_ = [e for e in expos if e["participant"] == p and e["round"] == r]
        by_round.append({"Seat": p, "Round": round_name(r), "Exposures": len(rows_),
                         "Realized P&L": sum(e["real_pnl"] for e in rows_),
                         "As taker": sum(e["real_pnl"] for e in rows_ if e["role"] == "initiator"),
                         "As quote owner": sum(e["real_pnl"] for e in rows_ if e["role"] == "quote_provider"),
                         "Own-info EV": sum((e["EV"] for e in rows_), Fraction(0))})
    assert sum(r_["Realized P&L"] for r_ in by_round if r_["Seat"] == p) == state["total_pnl"][p]
show(pd.DataFrame(by_round), signed=("Realized P&L", "As taker", "As quote owner", "Own-info EV"))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
width = 0.38
xs = range(1, ROUNDS + 1)
for ax, key, title in ((axes[0], "Realized P&L", "Realized P&L by trading round"),
                       (axes[1], "Own-info EV", "Own-information EV by trading round")):
    for i, p in enumerate(players):
        vals = [float(r_[key]) for r_ in by_round if r_["Seat"] == p]
        bars = ax.bar([x + (i - 0.5) * width for x in xs], vals, width=width * 0.92, color=SEAT_COLOR[p], label=p)
        ax.bar_label(bars, labels=[snum(v, 0) for v in vals], fontsize=8, color=INK2, padding=2)
    ax.axhline(0, color=NEUTRAL, linewidth=0.8)
    ax.set_xticks(list(xs), [f"R{r}\n{r - 1} shown" for r in xs])
    ax.set_title(title)
    ax.margins(y=0.15)
axes[0].legend(loc="best")
roles = [("initiator", "Taker"), ("quote_provider", "Quote owner")]
for i, p in enumerate(players):
    vals = [sum(e["real_pnl"] for e in expos if e["participant"] == p and e["role"] == role) for role, _ in roles]
    bars = axes[2].bar([j + (i - 0.5) * width for j in range(2)], vals, width=width * 0.92, color=SEAT_COLOR[p], label=p)
    axes[2].bar_label(bars, labels=[snum(v) for v in vals], fontsize=8, color=INK2, padding=2)
axes[2].axhline(0, color=NEUTRAL, linewidth=0.8)
axes[2].set_xticks([0, 1], [label for _, label in roles])
axes[2].set_title("Realized P&L by role")
axes[2].margins(y=0.15)
finish(fig)

grid = [[sum(e["real_pnl"] for e in expos if e["participant"] == A and e["hand"] == h and e["round"] == r)
         for h in range(1, N_HANDS + 1)] for r in range(1, ROUNDS + 1)]
lim = max(1, max(abs(v) for row in grid for v in row))
fig, ax = plt.subplots(figsize=(11, 1.2 + 0.45 * ROUNDS))
im = ax.imshow(grid, aspect="auto", cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim))
ax.set_yticks(range(ROUNDS), [f"R{r}" for r in range(1, ROUNDS + 1)])
ax.set_xticks(range(N_HANDS), [str(h) if h % 5 == 0 or h == 1 else "" for h in range(1, N_HANDS + 1)])
ax.set_xlabel("Hand")
ax.grid(False)
ax.set_title(f"{A} realized P&L by hand and round (blue = {A} gains, red = {B} gains)")
fig.colorbar(im, ax=ax, pad=0.01, fraction=0.03)
finish(fig)

## 5. Does decision quality improve as cards are revealed?

Each revealed community card narrows the range of possible settlement values. A good trader should therefore quote closer to the true value and take better-priced trades in later rounds. This section measures, for each seat and round:

* **Taker EV per fill.** The own-information EV of the buys and sells the seat sent; higher is better. *% +EV* is the share of those trades its own fair value favoured.
* **Quote error.** The mean distance from the seat's quote midpoint to the eventual settlement value V.
* **Fair-value error (baseline).** The same distance for the quoter's own fair value E[V], computed from the same cards. Quote error minus fair-value error is the **excess error**: how much worse the quote was than simply quoting E[V]. Positive excess error means the seat is not yet pricing in what it knows.
* **Spread.** The mean ask − bid width of the seat's quotes.

Each cell holds at most a few dozen observations from one match, so read these as patterns, not significance tests.

In [ ]:
quality = []
for p in players:
    for r in range(1, ROUNDS + 1):
        tk = [f for f in fills if f["participant"] == p and f["round"] == r]
        qs = [q for q in quotes if q["seat"] == p and q["round"] == r]
        quality.append({
            "Seat": p, "Round": round_name(r), "Taker fills": len(tk),
            "Taker EV / fill": mean(f["EV"] for f in tk), "% +EV": (100 * sum(f["EV"] > 0 for f in tk) / len(tk)) if tk else None,
            "Taker realized / fill": mean(Fraction(f["real_pnl"]) for f in tk),
            "Quotes": len(qs), "Quote error |mid−V|": mean(q["mid_err"] for q in qs),
            "Fair-value error |E[V]−V|": mean(q["fv_err"] for q in qs),
            "Excess error": (mean(q["mid_err"] for q in qs) - mean(q["fv_err"] for q in qs)) if qs else None,
            "Mean spread": mean(Fraction(q["spread"]) for q in qs),
        })
show(pd.DataFrame(quality), signed=("Taker EV / fill", "Taker realized / fill", "Excess error"))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for i, p in enumerate(players):
    rows_ = [q for q in quality if q["Seat"] == p]
    ev = [float(r_["Taker EV / fill"]) if r_["Taker EV / fill"] is not None else float("nan") for r_ in rows_]
    axes[0].bar([x + (i - 0.5) * width for x in xs], ev, width=width * 0.92, color=SEAT_COLOR[p], label=p)
    axes[1].plot(list(xs), [float(r_["Quote error |mid−V|"]) for r_ in rows_], color=SEAT_COLOR[p], marker="o", label=f"{p} quote mid")
    axes[1].plot(list(xs), [float(r_["Fair-value error |E[V]−V|"]) for r_ in rows_], color=SEAT_COLOR[p], marker="o",
                 markerfacecolor="white", linewidth=1, alpha=0.6, label=f"{p} own E[V] (baseline)")
    axes[2].plot(list(xs), [float(r_["Mean spread"]) for r_ in rows_], color=SEAT_COLOR[p], marker="o", label=p)
axes[0].axhline(0, color=NEUTRAL, linewidth=0.8)
axes[0].set_title("Taker EV per fill")
axes[0].legend()
axes[1].set_title("Distance to settlement value")
axes[1].legend(fontsize=7.5)
axes[1].set_ylim(bottom=0)
axes[2].set_title("Quoted spread (ask − bid)")
axes[2].set_ylim(bottom=0)
for ax in axes:
    ax.set_xticks(list(xs), [f"R{r}\n{r - 1} shown" for r in xs])
finish(fig)

display(Markdown("### Across the match\n\nRolling means over each seat's last ten fills as taker, in match order. "
                 "If a seat adapts to its opponent over the match, its taker EV per fill should trend upward."))
fig, ax = plt.subplots(figsize=(10, 3.2))
for p in players:
    seq_ = [f for f in fills if f["participant"] == p]
    roll = [float(mean(g["EV"] for g in seq_[max(0, k - 9):k + 1])) for k in range(len(seq_))]
    ax.plot([f["hand"] + (f["round"] - 1) / ROUNDS for f in seq_], roll, color=SEAT_COLOR[p], label=p)
ax.axhline(0, color=NEUTRAL, linewidth=0.8)
ax.set_xlabel("Hand")
ax.set_ylabel("Taker EV / fill (rolling 10)")
ax.set_title("Taker EV per fill over the match")
ax.legend()
finish(fig)

## 6. Comparison against fair value

A seat's **fair value** E[V] is the expected settlement value given only what that seat knew when it acted: its own private cards plus the community cards revealed so far. The opponent's cards and quotes are left out. This is the same baseline used for the EV columns. (This section is our reading of the requested "comparison against CV": every decision is compared with the fair value implied by the cards in hand.)

* **Taker decisions.** A buy is consistent with fair value when the ask is below E[V]. A sell is consistent when the bid is above E[V].
* **Quotes.** A quote is centred when its bid–ask range contains the quoter's E[V]. *Skew* is the quote midpoint minus E[V].

In the charts, points on the diagonal are priced exactly at fair value.

In [ ]:
fv = []
for p in players:
    tk = [f for f in fills if f["participant"] == p]
    agree = [f for f in tk if f["EV"] > 0]
    against = [f for f in tk if f["EV"] < 0]
    qs = [q for q in quotes if q["seat"] == p]
    fq = [q for q in qs if q["filled"]]
    fv.append({
        "Seat": p, "Taker fills": len(tk),
        "Consistent with E[V]": len(agree), "Against E[V]": len(against), "At E[V]": len(tk) - len(agree) - len(against),
        "Realized on consistent": sum(f["real_pnl"] for f in agree), "Realized on against": sum(f["real_pnl"] for f in against),
        "Quotes": len(qs), "% quotes containing E[V]": 100 * sum(q["straddles_E"] for q in qs) / len(qs) if qs else None,
        "Mean skew (mid − E[V])": mean(q["mid_vs_E"] for q in qs),
        "Mean |skew|": mean(abs(q["mid_vs_E"]) for q in qs),
        "Filled quotes": len(fq), "Quote-owner EV": sum((e["EV"] for e in expos if e["participant"] == p and e["role"] == "quote_provider"), Fraction(0)),
    })
show(pd.DataFrame(fv), signed=("Realized on consistent", "Realized on against", "Mean skew (mid − E[V])", "Quote-owner EV"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
lo = min(min(float(f["E"]) for f in fills), min(f["price"] for f in fills), min(float(q["E"]) for q in quotes), min(q["bid"] for q in quotes)) - 2
hi = max(max(float(f["E"]) for f in fills), max(f["price"] for f in fills), max(float(q["E"]) for q in quotes), max(q["ask"] for q in quotes)) + 2
for p in players:
    for side in ("buy", "sell"):
        pts = [f for f in fills if f["participant"] == p and f["side"] == side]
        axes[0].scatter([float(f["E"]) for f in pts], [f["price"] for f in pts], marker=MARK[side], s=42,
                        color=SEAT_COLOR[p], edgecolor="white", linewidth=1, label=f"{p} {side}s")
    qs = [q for q in quotes if q["seat"] == p]
    axes[1].vlines([float(q["E"]) for q in qs], [q["bid"] for q in qs], [q["ask"] for q in qs],
                   color=SEAT_COLOR[p], alpha=0.25, linewidth=1)
    axes[1].scatter([float(q["E"]) for q in qs], [float(q["mid"]) for q in qs], s=18, color=SEAT_COLOR[p],
                    edgecolor="white", linewidth=0.8, label=f"{p} quote mid (line = bid–ask)")
for ax in axes:
    ax.plot([lo, hi], [lo, hi], color=NEUTRAL, linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_xlabel("Actor's fair value E[V] at the time")
    ax.legend(fontsize=7.5, loc="upper left")
axes[0].set_ylabel("Fill price")
axes[0].set_title("Taker fills: price vs own fair value\n(buys below the line, sells above it, are consistent)")
axes[1].set_ylabel("Quote")
axes[1].set_title("Quotes vs the quoter's own fair value")
finish(fig)

## 7. EV versus realized P&L: successful submitted buy/sell actions

The rows below cover only successful buy and sell actions that the seat itself submitted. Quotes and improvements that were never filled have no stand-alone payoff and are left out.

* For a **buy** at price P, EV = E[V] − P and realized = V − P.
* For a **sell**, EV = P − E[V] and realized = P − V.

The **residual** is EV − realized. It is summed separately for each seat in match order. The residual combines card outcomes that were unknown at the fill with whatever the baseline ignores (the opponent's quotes, actions, and inventory). **It is not a measure of pure luck or of skill.** Negative EV on its own does not prove an error. Values are exact fractions until shown; tables show six decimal places.

In [ ]:
totals_rows = [{"Seat": p, "Contracts": at[p]["count"], "EV": Fraction(at[p]["ev_exact"]), "Realized": at[p]["real_pnl"],
                "EV − realized": Fraction(at[p]["ev_minus_pnl_exact"]),
                "EV (exact fraction)": at[p]["ev_exact"]} for p in players]
show(pd.DataFrame(totals_rows), places=6, signed=("EV", "Realized", "EV − realized"))

fig, axes = plt.subplots(1, len(players) + 1, figsize=(12.5, 3.6))
for ax, p in zip(axes, players):
    seq_ = [f for f in fills if f["participant"] == p]
    cev, creal, xs_ = Fraction(0), 0, list(range(1, len(seq_) + 1))
    evs, reals = [], []
    for f in seq_:
        cev += f["EV"]
        creal += f["real_pnl"]
        evs.append(float(cev))
        reals.append(creal)
    ax.plot(xs_, evs, color=SEAT_COLOR[p], label="cumulative EV")
    ax.plot(xs_, reals, color=NEUTRAL, label="cumulative realized")
    ax.axhline(0, color=NEUTRAL, linewidth=0.6)
    ax.set_title(f"{p}: submitted buy/sell actions")
    ax.set_xlabel("Fill # (as taker)")
    ax.legend(fontsize=8)
for p in players:
    seq_ = [f for f in fills if f["participant"] == p]
    axes[-1].plot(range(1, len(seq_) + 1), [float(f["CRES"]) for f in seq_], color=SEAT_COLOR[p], label=p)
axes[-1].axhline(0, color=NEUTRAL, linewidth=0.6)
axes[-1].set_title("Cumulative EV − realized")
axes[-1].set_xlabel("Fill # (as taker)")
axes[-1].legend(fontsize=8)
finish(fig)

ev_cols = {"sequence": "Seq", "hand": "Hand", "round": "Round", "side": "Side", "price": "Price", "E": "E[V]",
           "settlement_value": "V", "EV": "EV", "real_pnl": "Realized", "RES": "EV − realized",
           "CRES": "Cumulative EV − realized"}
for p in players:
    display(Markdown(f"### {p}: {at[p]['count']} submitted buy/sell actions"))
    show(pd.DataFrame([{v: f[k] for k, v in ev_cols.items()} for f in fills if f["participant"] == p]),
         places=6, signed=("EV", "Realized", "EV − realized", "Cumulative EV − realized"))

## 8. Both-party exposure (including passive quote-owner fills)

This table is separate from §7. It lists both sides of every fill: the taker's row and the quote owner's row. The quote owner's information is its own private cards plus the community cards revealed before the fill. Each seat's realized total must equal its final P&L; this is checked in the cell below.

In [ ]:
bp_rows = [{"Seat": p, "Exposures": bt[p]["count"],
            "As taker": sum(e["participant"] == p and e["role"] == "initiator" for e in expos),
            "As quote owner": sum(e["participant"] == p and e["role"] == "quote_provider" for e in expos),
            "EV": Fraction(bt[p]["ev_exact"]), "Realized": bt[p]["real_pnl"], "Final P&L": state["total_pnl"][p],
            "Reconciles": "✓" if bt[p]["real_pnl"] == state["total_pnl"][p] else "✗",
            "EV − realized": Fraction(bt[p]["ev_minus_pnl_exact"])} for p in players]
show(pd.DataFrame(bp_rows), places=6, signed=("EV", "Realized", "Final P&L", "EV − realized"))
assert all(r_["Reconciles"] == "✓" for r_ in bp_rows)

bp_cols = {"sequence": "Seq", "hand": "Hand", "round": "Round", "participant": "Seat", "role": "Role",
           "initiator": "Taker", "side": "Side", "price": "Price", "E": "E[V]", "settlement_value": "V",
           "EV": "EV", "real_pnl": "Realized", "cumulative_ev_minus_pnl": "Cumulative EV − realized (seat)"}
bp_df = pd.DataFrame([{v: (Fraction(e["cumulative_ev_minus_pnl_exact"]) if k == "cumulative_ev_minus_pnl" else e[k])
                       for k, v in bp_cols.items()} for e in expos])
bp_df["Role"] = bp_df["Role"].map({"initiator": "taker", "quote_provider": "quote owner"})
show(bp_df, places=6, signed=("EV", "Realized", "Cumulative EV − realized (seat)"),
     collapsed=f"<b>All {len(expos)} exposure rows</b> (click to expand)")

## 9. Time to decide

Timing comes from the seat transcripts.

* **Model response time** runs from the moment a model request is sent to the moment its response, containing the action, comes back. This is the time the agent took to decide. If one response contains several actions, they share that time.
* **Wake to first action** runs from the environment's wake message to the first environment action the seat sent in response.

The harness's own `time_to_action` counters from `run.json` are shown for comparison. All times are wall-clock and include provider latency.

In [ ]:
if not len(tools_df):
    display(Markdown("_No seat transcripts available: timing cannot be reported for this run._"))
else:
    act = tools_df[tools_df["action"] != "say"]
    tta = run_status.get("time_to_action") or {}
    trows = []
    for p in telemetry_seats:
        a_ = act[act["seat"] == p]["latency_s"]
        w_ = [(w["first_action_ms"] - w["t"]) / 1000 for w in wakes if w["seat"] == p and w["first_action_ms"]]
        h_ = tta.get(p) or {}
        trows.append({"Seat": p, "Action calls": len(a_), "Median response (s)": a_.median(), "Mean (s)": a_.mean(),
                      "p90 (s)": a_.quantile(0.9), "Max (s)": a_.max(),
                      "Wakes with an action": len(w_), "Median wake→action (s)": statistics.median(w_) if w_ else None,
                      "Harness time_to_action mean (s)": h_["total_ms"] / h_["count"] / 1000 if h_.get("count") else None,
                      "Harness max (s)": h_.get("max_ms", 0) / 1000 if h_ else None})
    show(pd.DataFrame(trows))

    by_action = (act.groupby(["seat", "action"])["latency_s"].agg(["count", "median", "mean", "max"])
                 .reset_index().rename(columns={"seat": "Seat", "action": "Action", "count": "Calls",
                                                "median": "Median (s)", "mean": "Mean (s)", "max": "Max (s)"}))
    show(by_action)

    order = [a for a in ("open_market", "improve_market", "buy", "sell", "next_hand") if a in set(act["action"])]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for i, p in enumerate(telemetry_seats):
        data = [act[(act["seat"] == p) & (act["action"] == a)]["latency_s"].tolist() or [float("nan")] for a in order]
        pos = [j + (i - 0.5) * 0.38 for j in range(len(order))]
        bp = axes[0].boxplot(data, positions=pos, widths=0.32, patch_artist=True, showfliers=True,
                             medianprops={"color": "white", "linewidth": 1.5},
                             flierprops={"marker": "o", "markersize": 3, "markerfacecolor": SEAT_COLOR[p], "markeredgecolor": "none"},
                             whiskerprops={"color": SEAT_COLOR[p]}, capprops={"color": SEAT_COLOR[p]})
        for box in bp["boxes"]:
            box.set(facecolor=SEAT_COLOR[p], edgecolor=SEAT_COLOR[p])
        axes[0].plot([], [], color=SEAT_COLOR[p], linewidth=6, label=p)
        tr = act[(act["seat"] == p) & act["round"].notna()]
        med = tr[tr["action"].isin(["open_market", "improve_market", "buy", "sell"])].groupby("round")["latency_s"].median()
        axes[1].plot(med.index, med.values, color=SEAT_COLOR[p], marker="o", label=p)
        s_ = act[act["seat"] == p].sort_values("end_ms")
        axes[2].scatter(s_["hand"], s_["latency_s"], s=10, color=SEAT_COLOR[p], alpha=0.35, edgecolor="none")
        roll = s_.groupby("hand")["latency_s"].median().rolling(3, min_periods=1).median()
        axes[2].plot(roll.index, roll.values, color=SEAT_COLOR[p], label=f"{p} (rolling median)")
    axes[0].set_xticks(range(len(order)), [o.replace("_", "\n") for o in order])
    axes[0].set_ylabel("Model response time (s)")
    axes[0].set_title("Decision time by action")
    axes[0].legend(fontsize=8)
    axes[1].set_xticks(list(xs), [f"R{r}\n{r - 1} shown" for r in xs])
    axes[1].set_title("Median trading-decision time by round")
    axes[1].set_ylim(bottom=0)
    axes[1].legend(fontsize=8)
    axes[2].set_xlabel("Hand")
    axes[2].set_title("Decision time over the match")
    axes[2].legend(fontsize=8)
    finish(fig)

    by_outcome = (tools_df.groupby(["seat", "outcome"])["latency_s"].agg(["count", "median"]).reset_index()
                  .rename(columns={"seat": "Seat", "outcome": "Outcome", "count": "Tool calls", "median": "Median response (s)"}))
    display(Markdown("Model response time by tool-call outcome. *harness: …* calls never reached the environment "
                     "(*stale_view*: the state had changed since the seat last observed it)."))
    show(by_outcome)

## 10. Cost and token usage

Usage and cost are the per-request fields the harness recorded in each seat transcript: input, cached input (cache read), output, and the cost computed from the model catalog's prices at run time. The provider's actual bill may differ. Each request is assigned to a hand by its completion time (a hand ends when its last fill commits). The run's host session made no model requests, so the seat totals below are the whole cost of playing the game.

In [ ]:
if not len(calls_df):
    display(Markdown("_No seat transcripts available: usage cannot be reported for this run._"))
else:
    crows = []
    for p in telemetry_seats + ["Total"]:
        d = calls_df if p == "Total" else calls_df[calls_df["seat"] == p]
        n_fill = len(fills) if p == "Total" else sum(e["participant"] == p for e in expos)
        crows.append({"Seat": p, "Model requests": len(d), "Requests with tool calls": int((d["n_tools"] > 0).sum()),
                      "Input tokens": int(d["input"].sum()), "Cached input": int(d["cache_read"].sum()),
                      "Output tokens": int(d["output"].sum()), "Total tokens": int(d["total_tokens"].sum()),
                      "Mean input / request": d["input"].mean(), "Max input / request": int(d["input"].max()),
                      "Cost (USD)": f"${d['cost'].sum():.4f}", "Cost / hand": f"${d['cost'].sum() / N_HANDS:.4f}",
                      "Cost / fill exposure": f"${d['cost'].sum() / n_fill:.4f}",
                      "Aborted / errored requests": int((~d["stop"].isin(["toolUse", "stop"])).sum())})
    show(pd.DataFrame(crows), places=0)
    display(Markdown(f"**Total recorded model cost for the game: ${calls_df['cost'].sum():.4f}** "
                     f"({int(calls_df['total_tokens'].sum()):,} tokens across {len(calls_df)} requests)."))

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for p in telemetry_seats:
        d = calls_df[calls_df["seat"] == p].sort_values("end_ms")
        if "hand" in d:
            per_hand = d.groupby("hand")["cost"].sum().reindex(range(1, N_HANDS + 1), fill_value=0)
            axes[0].plot(per_hand.index, per_hand.cumsum().values, color=SEAT_COLOR[p], label=p)
            tok = d.groupby("hand")["total_tokens"].sum().reindex(range(1, N_HANDS + 1), fill_value=0)
            axes[1].plot(tok.index, tok.values / 1000, color=SEAT_COLOR[p], marker="o", markersize=3, label=p)
        axes[2].scatter(range(1, len(d) + 1), d["input"] / 1000, s=8, color=SEAT_COLOR[p], alpha=0.6, edgecolor="none", label=p)
    axes[0].set_title("Cumulative cost (USD)")
    axes[0].set_xlabel("Hand")
    axes[1].set_title("Tokens per hand (thousands)")
    axes[1].set_xlabel("Hand")
    axes[1].set_ylim(bottom=0)
    axes[2].set_title("Input tokens per request (thousands)")
    axes[2].set_xlabel("Request # (per seat)")
    axes[2].set_ylim(bottom=0)
    for ax in axes:
        ax.legend(fontsize=8)
    finish(fig)

## 11. Action mix and rejections

The **environment** counts every invoke attempt that reached it: accepted or rejected, each with a receipt. The **harness** may also stop a tool call before it reaches the environment: a stale view, an unknown tool name, or a host error. Those calls have no receipt and are counted separately here. They are not part of the §12 action table.

In [ ]:
mix = []
for p in players:
    mine = [a for a in actions if a["actor"] == p]
    row = {"Seat": p}
    for kind in ("open_market", "improve_market", "buy", "sell", "next_hand"):
        row[kind] = sum(a["success"] and a["action"] == kind for a in mine)
    row["Rejected by environment"] = sum(not a["success"] for a in mine)
    row["Stopped by harness"] = int(((tools_df["seat"] == p) & tools_df["outcome"].astype(str).str.startswith("harness")).sum()) if len(tools_df) else 0
    row["Public say"] = sum(c["participant"] == p for c in chats)
    mix.append(row)
mix_df = pd.DataFrame(mix)
show(mix_df)

cats = [c for c in mix_df.columns if c != "Seat"]
fig, ax = plt.subplots(figsize=(10, 0.5 + 0.55 * len(cats)))
h_ = 0.38
for i, p in enumerate(players):
    vals = [mix_df.loc[mix_df["Seat"] == p, c].iloc[0] for c in cats]
    bars = ax.barh([j + (i - 0.5) * h_ for j in range(len(cats))], vals, height=h_ * 0.92, color=SEAT_COLOR[p], label=p)
    ax.bar_label(bars, fontsize=8, color=INK2, padding=2)
ax.set_yticks(range(len(cats)), cats)
ax.invert_yaxis()
ax.set_title("Tool calls by type and outcome")
ax.legend()
finish(fig)

rej = Counter((a["actor"], a["action"], str(a["result"])) for a in actions if not a["success"])
show(pd.DataFrame([{"Seat": s, "Attempted action": act_, "Environment error": err, "Count": n}
                   for (s, act_, err), n in sorted(rej.items())]))

## 12. Complete action table

This table lists every invoke attempt in receipt order (accepted and rejected), grouped by hand. Expand a hand to see its rows.

* **Before** is the acting seat's view just before the attempt: the board, its own cards, the live market, positions, its legal actions, and its own fair value E[V].
* **Result** is the environment's event text for an accepted action, or the error for a rejected one.
* **After** is the state following the action. A rejected attempt leaves the state unchanged; the replay checked this.

Each row has a nested *details* toggle with cash, cumulative P&L, both seats' legal actions, the range of possible settlement values, and engine revisions. The states are reconstructed by the replay; they are not verbatim tool responses. Rounds and hands are numbered from 1.

In [ ]:
def legal_from(st, p):
    """Legal actions for seat p in a committed engine state (same rule as the replay)."""
    if st["phase"] == "settled":
        return ["next_hand"] if p == players[0] else []
    if st["phase"] != "trading" or st["active"] != p:
        return []
    m = st["market"]
    if m is None:
        return ["open_market"]
    return (["buy"] if m["ask_owner"] != p else []) + (["sell"] if m["bid_owner"] != p else []) + \
        (["improve_market"] if m["ask"] - m["bid"] > 1 else [])


def market_text(m):
    if not m:
        return "none"
    owner = m["bid_owner"] if m["bid_owner"] == m["ask_owner"] else f"{m['bid_owner']}/{m['ask_owner']}"
    return f"{m['bid']} / {m['ask']} ({owner})"


def state_cell(view, actor):
    pos = ", ".join(f"{q} {snum(v, 0)}" for q, v in view["positions"].items())
    return (f"<b>board</b> {' '.join(view['community_cards']) or '—'}<br><b>own</b> {' '.join(view['your_private_cards'])}"
            f"<br><b>market</b> {market_text(view['market'])}<br><b>positions</b> {pos}")


def esc(x):
    return html.escape(str(x))


head = ("<tr><th>Seq</th><th>Rnd</th><th>Actor</th><th>Action</th><th>OK</th><th>Before (actor's view)</th>"
        "<th>Own E[V]</th><th>Result / error</th><th>After</th><th>More</th></tr>")
blocks = []
by_hand = defaultdict(list)
for a in actions:
    by_hand[a["hand_before"]].append(a)
for h in sorted(by_hand):
    rows_html = []
    for a in by_hand[h]:
        before, after = a["state_before"], a["state_after"]
        eb, ea = a["committed_engine_before"], a["committed_engine_after"]
        args = " ".join(f"{k}={v}" for k, v in a["arguments"].items())
        legal_b = ", ".join(before["available_actions"]) or "none"
        result = a["result"]["event"] if a["success"] and isinstance(a["result"], dict) else a["result"]
        after_cell = state_cell(after, a["actor"]) if a["success"] else "<i>unchanged</i>"
        more = ("<details><summary>details</summary>"
                f"<b>phase</b> {esc(eb['phase'])} → {esc(ea['phase'])}<br>"
                f"<b>cash before</b> {esc(', '.join(f'{q} {v}' for q, v in eb['cash'].items()))}<br>"
                f"<b>cash after</b> {esc(', '.join(f'{q} {v}' for q, v in ea['cash'].items()))}<br>"
                f"<b>cumulative P&amp;L</b> {esc(', '.join(f'{q} {v}' for q, v in before['total_pnl'].items()))}<br>"
                f"<b>trades this round</b> {eb['trades_this_round']} → {ea['trades_this_round']}<br>"
                f"<b>legal before</b> {esc('; '.join(f'{q}: {legal_from(eb, q) or []}' for q in players))}<br>"
                f"<b>legal after</b> {esc('; '.join(f'{q}: {legal_from(ea, q) or []}' for q in players))}<br>"
                f"<b>possible V</b> {a['possible_value_min']}–{a['possible_value_max']} (E[V] exact {esc(a['own_card_expected_value_exact'])})<br>"
                f"<b>revision</b> {eb['revision']} → {ea['revision']}</details>")
        rows_html.append(
            f"<tr><td>{a['sequence']}</td><td>R{a['round_before']}</td><td>{esc(a['actor'])}</td>"
            f"<td>{esc(a['action'])} {esc(args)}</td><td>{'✓' if a['success'] else '✗'}</td>"
            f"<td style='text-align:left'>{state_cell(before, a['actor'])}<br><b>legal</b> {esc(legal_b)}</td>"
            f"<td>{num(Fraction(a['own_card_expected_value_exact']))}</td><td style='text-align:left'>{esc(result)}</td>"
            f"<td style='text-align:left'>{after_cell}</td><td style='text-align:left'>{more}</td></tr>")
    n_ok = sum(a["success"] for a in by_hand[h])
    settled = history[h - 1] if h <= N_HANDS else None
    tail = (f" · value {settled['value']} · " + ", ".join(f"{q} {snum(settled['pnl'][q])}" for q in players)) if settled else ""
    blocks.append(f"<details{' open' if h == 1 else ''}><summary><b>Hand {h}</b>: {len(by_hand[h])} attempts "
                  f"({n_ok} accepted, {len(by_hand[h]) - n_ok} rejected){esc(tail)}</summary>"
                  f"<table class='dataframe' border='0'><thead>{head}</thead><tbody>{''.join(rows_html)}</tbody></table></details>")
display(Markdown(f"**{len(actions)} invoke attempts: {sum(a['success'] for a in actions)} accepted, "
                 f"{sum(not a['success'] for a in actions)} rejected.**"))
display(HTML("".join(blocks)))

## 13. Public chat

Public `say` calls are counted separately from invoke attempts. A `say` (`handle_input`) sends a public notice and adds one chat event to the shared public feed. It does not change the market state, advance a turn, or create a fill. Only the public chat text is shown here, never assistant replies or reasoning. Empty or whitespace-only messages are labelled as such so that they remain visible.

In [ ]:
blank = sum(not c["text"].strip() for c in chats)
display(Markdown(f"**{len(chats)} public say calls** "
                 f"({', '.join(f'{p} {sum(c['participant'] == p for c in chats)}' for p in players)}), "
                 f"separate from {len(actions)} invoke attempts. {blank} of {len(chats)} contain no visible text."))
chat_rows = []
for c in chats:
    prog = c["state_before"]["progress"]
    text = c["text"]
    chat_rows.append({"Seq": c["sequence"], "Seat": c["participant"],
                      "Hand": prog["hands_settled"] + 1 if c["state_before"]["phase"] == "trading" else f"{prog['hands_settled']} (settled)",
                      "Round": prog["trading_round"], "Accepted": "✓" if c["success"] else "✗",
                      "Public text": text if text.strip() else f"(blank: {len(text)} whitespace character{'s' if len(text) != 1 else ''})"})
if chat_rows:
    show(pd.DataFrame(chat_rows))

## 14. Verification results

Every check below has already passed as an `assert` in the cells above; this cell would not run otherwise. The table records the checks and their values. It follows the format of the existing report's `verification.md`.

In [ ]:
per_hand_ok = all(len(set(sum(h["cards"].values(), [])) | set(h["board"])) == HAND_CARDS and
                  sum(f["hand"] == h["hand"] for f in fills) == ROUNDS * cfg["trades_per_round"] for h in history)
exact_ok = all(Fraction(at[p]["ev_exact"]) - at[p]["real_pnl"] == Fraction(at[p]["ev_minus_pnl_exact"]) and
               sum((f["RES"] for f in fills if f["participant"] == p), Fraction(0)) == Fraction(at[p]["ev_minus_pnl_exact"])
               for p in players)
checks = [
    ("Receipt replay", f"Sorted replay from the first deal reproduces checkpoint.snapshot exactly (revision {snapshot['revision']}, "
                       "history, events, cash, positions, final settlement)", True),
    ("Legality", f"All {sum(a['success'] for a in actions)} accepted actions legal for their actor; quote integers/order, "
                 "strict non-worsening improvements, owner/side and fill prices pass", True),
    ("Rejections", f"All {sum(not a['success'] for a in actions)} rejected attempts preserve committed engine state", True),
    ("Deals", f"{N_HANDS} hands, each with {HAND_CARDS} distinct cards and {ROUNDS * cfg['trades_per_round']} fills", per_hand_ok),
    ("Settlements", f"Rank sums, cash + position × value, hand P&L and cumulative totals match all {N_HANDS} settlements", True),
    ("Final P&L", " / ".join(f"{p} {snum(state['total_pnl'][p])}" for p in players), True),
    ("Wake states", f"{wakecount} transcript wake states match own cards, board prefix and cumulative P&L", not wake_problems),
    ("Primary actor totals", "; ".join(f"{p} EV {float(Fraction(at[p]['ev_exact'])):.12f}, realized {at[p]['real_pnl']} "
                                       f"({at[p]['count']})" for p in players), True),
    ("Both-party totals", "; ".join(f"{p} EV {float(Fraction(bt[p]['ev_exact'])):.12f}, realized {bt[p]['real_pnl']} "
                                    f"({bt[p]['count']} each)" for p in players) + "; realized = final P&L",
     all(bt[p]["real_pnl"] == state["total_pnl"][p] for p in players)),
    ("Round reconciliation", "Both-party realized P&L summed by round equals final P&L for each seat", True),
    ("Exact arithmetic", "EV, residual and per-seat cumulative residual verified as exact Fractions", exact_ok),
    ("Information set", "E[V] uses own private cards + community cards revealed before the fill only", True),
    ("Market opens", "; ".join(f"{opens_by_seat[p]} {p}" for p in players), True),
    ("Invoke / chat / observe", f"{len(actions)} invoke attempts, {len(chats)} public say calls, {n_observe} observe calls", True),
    ("Transcript alignment", f"{sum(t['sequence'] is not None for t in tool_calls)} receipts matched to transcript tool calls "
                             f"(action + arguments); {sum(str(t['outcome']).startswith('harness') for t in tool_calls)} "
                             "harness-stopped calls had no receipt", bool(tool_calls) or not telemetry_seats),
    ("Privacy", "No raw reasoning or assistant text displayed; run must be finished before any private card is shown", True),
]
show(pd.DataFrame([{"Check": c, "Detail": d, "Result": "PASS" if ok else "FAIL"} for c, d, ok in checks]))
assert all(ok for _, _, ok in checks)

## 15. Methods and limitations

**Source of truth.** All game facts come from a deterministic replay of the run's own `checkpoint.json` receipts. The rebuilt final state must equal the saved snapshot exactly. No game is started, resumed, or modified, and nothing is written to disk. The replay, wake verification, and EV accounting cells are reused from the existing generator (`report.ipynb`). The replay cell's only change turns a failed replay into a clear rejection.

**Fair value.** Let k be the number of cards the actor knows (its own private cards plus the community cards revealed before the fill) and S their rank sum. Let N be the number of cards in the settlement value (twice the private cards per player, plus the community cards) and T = 364, the rank sum of a full deck. The remaining N − k cards are drawn without replacement from the 52 − k unseen cards, so

`E[V] = S + (N − k)(T − S) / (52 − k)`.

This baseline excludes the opponent's cards and every signal in the opponent's quotes and actions. Subjective EVs therefore differ between seats and need not sum to zero.

**EV and residual.** A buy at price P has EV E[V] − P and realized payoff V − P; a sell is the mirror image. Every contract is valued at its own hand's settlement, even when a later contract offsets it. The residual (EV − realized) mixes card outcomes that were unknown at the fill with information the baseline ignores. It is **not** a measure of pure luck or of skill.

**Rounds.** Trading round r takes place with r − 1 community cards showing. The final reveal settles the hand, so there is no trading round after the last card.

**Timing and cost.** Times come from the transcript timestamps and include network and provider latency. Cost is the per-request figure recorded by the harness from its model price catalog, not an invoice. Requests are assigned to hands by completion time.

**Limitations.**

* This is one match between two seats. Per-round statistics rest on a few dozen observations and are descriptive, not significance tests.
* The pre- and post-action states in §12 are reconstructed by the replay, not copied from the tool responses.
* The recent-event feed is limited to 30 events by design.
* Stale-view calls are known only from the transcripts.
* Public chat text is shown exactly as sent. Private reasoning is never shown.